In [ ]:
print("ok")

ok


In [ ]:
%pwd

'c:\\Users\\ayush\\GenAI_projects2\\medical_chatbot\\research'

In [ ]:
import os
os.chdir("../")

In [ ]:
%pwd

'c:\\Users\\ayush\\GenAI_projects2\\medical_chatbot'

In [ ]:
from langchain.document_loaders import PyPDFLoader , DirectoryLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
#extract data from pdf file
def load_pdf_file(data):
    loader= DirectoryLoader(data,
                            glob="*.pdf",
                            loader_cls=PyPDFLoader
                            )
    documents=loader.load()
    return documents

In [ ]:
extracted_data=load_pdf_file(data='Data/')

In [ ]:
# extracted_data

In [ ]:
#split data into smaller chunks
def text_split(extracted_data):
    text_splitter=RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)
    text_chunks=text_splitter.split_documents(extracted_data)
    return text_chunks

In [ ]:
text_chunks=text_split(extracted_data)
print("Length of Text Chunks",len(text_chunks))

Length of Text Chunks 5859


In [ ]:
# text_chunks

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from sentence_transformers import SentenceTransformer

In [ ]:
#download the huggingface embeddings model
def download_huggingface_embeddings():
    embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
    return embeddings

In [ ]:
embeddings=download_huggingface_embeddings()
print(type(embeddings))

C:\Users\ayush\AppData\Local\Temp\ipykernel_7188\2159769555.py:3: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings=HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')


<class 'langchain_community.embeddings.huggingface.HuggingFaceEmbeddings'>


In [ ]:
query_result=embeddings.embed_query("Hello world")
print("Length",len(query_result))

Length 384


In [ ]:
# query_result

In [ ]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [ ]:
PINECONE_API_KEY= os.environ.get("PINECONE_API_KEY")

In [ ]:
from pinecone import Pinecone

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("medical-chatbot")


In [ ]:
os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY

In [ ]:
# #embed each chunk and upsert the embeddings into your Pinecone index
# from langchain_pinecone import PineconeVectorStore
# docsearch = PineconeVectorStore.from_documents(
#     documents=text_chunks,
#     index_name="medical-chatbot" ,
#     embedding=embeddings,
# )

In [ ]:
#load existing index
from langchain_pinecone import PineconeVectorStore
#embed each chunk and upsert ther embeddings into your Pinecone index
docsearch = PineconeVectorStore.from_existing_index(
    index_name="medical-chatbot",
    embedding=embeddings,
)


In [ ]:
docsearch

In [ ]:
retriever=docsearch.as_retriever(search_type="similarity",search_kwargs={"k":3})

In [ ]:
retrieved_docs=retriever.invoke("what is acne")

In [ ]:
retrieved_docs

[Document(id='92ad63bf-2986-4e51-b1de-d8aaf6a94cb2', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='fc0047c2-0560-4d83-ab2d-28abca2f1274', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 'page_label': '40', 'producer': 'PDFlib+PDI 5.0.0 (SunOS)', 'source': 'Data\\Medical_book.pdf', 'total_pages': 637.0}, page_content='GALE ENCYCLOPEDIA OF MEDICINE 226\nAcne\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 26'),
 Document(id='27e9f374-ecbc-411b-9cf7-758dfa7ce2a1', metadata={'creationdate': '2004-12-18T17:00:02-05:00', 'creator': 'PyPDF', 'moddate': '2004-12-18T16:15:31-06:00', 'page': 39.0, 

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain
from langchain.llms import CTransformers
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

llm = CTransformers(
    model=r'C:\Users\ayush\LLM_models\models',
    model_type='llama',
    config={
        'temperature': 0.01,
        'max_new_tokens': 256
    }
)

In [61]:
system_prompt=(
    "You are an assistant for question-answering tasks."
    "Use the following pieces of retrieved context to answer"
    "the question. If you don't know the answer, say that you"
    "don't know. Use three sentences maximum and keep the "
    "answer concise."
    "\n\n"
    "{context}"
)
prompt= ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}"),
    ]
)

In [ ]:
question_answer_chain= create_stuff_documents_chain(llm,prompt)
rag_chain= create_retrieval_chain(retriever,question_answer_chain)

In [63]:
response= rag_chain.invoke({"input":"What is Acne?"})
print(response["answer"])


Assistant: Acne is a common skin condition characterized by inflammation of the sebaceous glands, which can lead to pimples, blackheads, and whiteheads on the face, back, and chest. It is caused by hormonal changes, excess oil production, and bacterial infections. Treatment options include topical creams and oral antibiotics, as well as lifestyle changes such as regular exercise and a healthy diet.
